In [1]:
from os.path import join, dirname
import os
import sys
import random
from copy import deepcopy
from hydra import initialize, compose
from omegaconf import DictConfig
import numpy as np
import torchshow as ts
import torchvision

import torch
import torch.nn.functional as F
from einops import rearrange

sys.path.append(join(dirname(dirname(os.getcwd()))))
sys.path.append(os.getcwd())

from omniisaacgymenvs.utils.hydra_cfg.hydra_utils import *
from omniisaacgymenvs.utils.hydra_cfg.reformat import omegaconf_to_dict, print_dict
from omniisaacgymenvs.utils.task_util import initialize_task
from omniisaacgymenvs.envs.vec_env_rlgames import VecEnvRLGames

from memoryPerceiver.memory_perceiver.main.configuration import cf
import memoryPerceiver.memory_perceiver.isaac.trainer as tr
from memoryPerceiver.memory_perceiver.isaac.plot import Plotter
from memoryPerceiver.memory_perceiver.isaac.agent import ActorCritic, ActorCriticSimple, ActorCriticPerceiver

%load_ext autoreload
%autoreload 2
%env ISAAC_JUPYTER_KERNEL=1

env: ISAAC_JUPYTER_KERNEL=1


/isaac-sim/kit/python/lib/python3.7/site-packages/IPython/extensions/autoreload.py:121: DeprecationWarning: the imp module is deprecated in favour of importlib; see the module's documentation for alternative uses
  from imp import reload


In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device_cpu = torch.device("cpu")
random.seed(0)
np.random.seed(0)
torch.use_deterministic_algorithms(False)
%env CUBLAS_WORKSPACE_CONFIG=:4096:8

env: CUBLAS_WORKSPACE_CONFIG=:4096:8


In [3]:
"""model = ActorCriticSimple(cf, device)
model_student = ActorCriticSimple(cf, device)

#actions, logs, vals, dist = model.act(torch.randn(8,4, device=device))
#print(actions.size(), logs.size(), vals.size())
"""

'model = ActorCriticSimple(cf, device)\nmodel_student = ActorCriticSimple(cf, device)\n\n#actions, logs, vals, dist = model.act(torch.randn(8,4, device=device))\n#print(actions.size(), logs.size(), vals.size())\n'

In [4]:
model = ActorCriticPerceiver(cf, device)
model_student = ActorCriticPerceiver(cf, device)
"""
numEnvs=1
states = {'observations' : torch.randn(numEnvs,cf.frames,4, device=device),
         'video' : torch.zeros(numEnvs, 3, cf.frames, 256, 456, device=device),
         'audio':torch.zeros(numEnvs, cf.frames*1920, device=device)
        }
model.act(states)
"""

"\nnumEnvs=1\nstates = {'observations' : torch.randn(numEnvs,cf.frames,4, device=device),\n         'video' : torch.zeros(numEnvs, 3, cf.frames, 256, 456, device=device),\n         'audio':torch.zeros(numEnvs, cf.frames*1920, device=device)\n        }\nmodel.act(states)\n"

In [5]:
"""
do not care about memory being the same, backpropagate such that score are good
"""
#@hydra.main(config_name="config", config_path="../cfg")
#@hydra.main(config_name="config", config_path="../../OmniIsaacGymEnvs/omniisaacgymenvs/cfg")
def parse_hydra_configs(cfg: DictConfig):
    cfg_dict = omegaconf_to_dict(cfg)

    render = True
    env = VecEnvRLGames(headless=cfg.headless, sim_device=cfg.device_id, enable_viewport=cfg.enable_viewport)
    
    
    # !!! Load isaac modules after the variables
    from omni.isaac.core.utils.torch.maths import set_seed
    import omni.usd
    from omni.isaac.core.utils.extensions import enable_extension
    enable_extension("omni.isaac.sensor")
    from omni.isaac.core.utils.stage import open_stage, save_stage
    
    plotter = Plotter(titles=['loss_actor', 'loss_critic', 'rewards'])
    history_keys = ['actions', 'values', 'logprobs', 'returns', 'rewards', 'advantages', 'death_index', 'generator']
    history = tr.init_history(history_keys, sim_duration, numEnvs, cf.dim_actions_ppo, device)
    previous_history = deepcopy(history)
    buffer = {'video':[], 'audio':[], 'obs':[], 'rewards':[], 'actions_star':[]}
    memories = {i:None for i in range(numEnvs)}

    cfg.seed = set_seed(cfg.seed, torch_deterministic=cfg.torch_deterministic)
    cfg_dict['seed'] = cfg.seed  # sets seed. if seed is -1 will pick a random one
    task = initialize_task(cfg_dict, env)
    all_losses_actor, all_losses_critic, all_returns = [], [], []
    optimizer = torch.optim.Adam(model_student.parameters(), lr=learning_rate)
    sim_count, sim_step = -1, 0
    first_run=True
    is_inference=False  # Will switch to True immediately
    
    while env._simulation_app.is_running():
        if env._world.is_playing():
            # if env._world.current_time_step_index == 0 or \
            # env._world.current_time_step_index % sim_duration == 0:
            if env.sim_frame_count % sim_duration == 0:
                
                ### Plots

                if sim_count % 6 == 1 and len(all_losses_actor)>0:
                    plotter.plot_all([(all_losses_actor, 160), (all_losses_critic, 160), all_returns])

                ### Initialization

                sim_count+=1
                env._world.reset(soft=False)
                is_inference = not is_inference
                
                print('$$$$$$$$$$$ CUMUL REWARD $$$$$$$$$$$')
                if len(history['rewards'])>0:
                    print(history['rewards'].sum())
                if is_inference:
                    print('PLAY')
                    if first_run and os.path.exists(path_model):
                        model.load_state_dict(torch.load(path_model))
                        model_student.load_state_dict(torch.load(path_model))
                    else:
                        if sim_count % student_update_step*2 == 0:
                            model.load_state_dict(model_student.state_dict())
                    model.actor.reset_memories()
                    torch.save(model_student.state_dict(), path_model)
                    model.eval()
                else:
                    print('REPLAY')
                    first_run=False
                    model.actor.reset_memories()
                    model_student.actor.reset_memories()
                    model_student.train() 
                    optimizer.zero_grad()
                    previous_history = deepcopy(history)
                    previous_history['returns'] = tr.calculate_returns(history['rewards'], history['death_index'], gamma=c_gamma)
                    previous_history['advantages'] = tr.calculate_gae(history['rewards'], history['values'], 
                                                        history['death_index'], gamma=c_gamma, gae_lambda=c_lambda)
                    all_returns.append(previous_history['returns'][1, :].mean().item())
                    print(previous_history['returns'][:200, 1])
                    
                torch.manual_seed(sim_count//2)
                torch.cuda.manual_seed(sim_count//2)
                memories = {i:None for i in range(numEnvs)}
                full_video = []
                birth_time = torch.zeros(numEnvs)
                for key in buffer.keys():
                    buffer[key]=[]
                history = tr.init_history(history_keys, sim_duration, numEnvs, cf.dim_actions_ppo, device)
                backward_count, kl_div = 0, 0
                sim_step = 0

            ### Get agent observations 

            all_obs = env._task.get_observations()['articulation_view']
            obs = all_obs['obs_buf'].clone()    # (numEnvs, d)
            video = all_obs['video_buf'].clone()  # (numEnvs, 256, 456, 3)
            #video = torch.zeros(numEnvs, 256, 456, 3, device=device)
            #ts.save(video[0], '/isaac-sim/memoryPerceiver/memory_perceiver/main/test0.jpg')
            full_video.append(video.cpu())
            env._task.calculate_metrics()       
            reward=env._task.rew_buf.clone()    # (numEnvs, 1)
            tr.update_buffers(buffer, {'obs': obs, 'video': video, 'rewards': reward}, frames=cf.frames)
            history['rewards'][sim_step] = reward

            ### Reset memory at death and record 

            reset_indexes = []
            for agent_i in range(numEnvs):
                if env._task.reset_buf[agent_i].item() == 1:
                    reset_indexes.append(agent_i)
                    birth_time[agent_i] = sim_step

                    # Update death dictionnary during inference

                    if is_inference:
                        history['death_index'][sim_step, agent_i]=1
            model.actor.reset_memories(indexes=reset_indexes)
            if not is_inference:
                model_student.actor.reset_memories(indexes=reset_indexes)

            ### Agent
            
            if len(buffer['actions_star'])==0 :

                ## Process inputs

                states={}
                mask = tr.create_mask(birth_time, sim_step, frames=cf.frames)
                states['observations'] = tr.buffer_padding(torch.stack(buffer['obs']), [cf.dim_observations],
                                                            numEnvs=numEnvs, frames=cf.frames)
                states['observations'][mask] *= 0
                states['video'] = torch.zeros(numEnvs, cf.frames, 256, 456, 3, device=device)
                # states['video'] = tr.buffer_padding(torch.stack(buffer['video']), [256, 456, 3], numEnvs=numEnvs, frames=cf.frames)
                states['video'][mask] *= 0
                states['video'] = rearrange(states['video'], 'b t h w c -> b c t h w')
                states['audio'] = torch.zeros(numEnvs, cf.frames*1920, device=device)

                ## Inference, save history and memory update

                if is_inference:
                    actions, logprobs, values, old_dist = model.act(states)   
                    history['actions'][sim_step] = actions  
                    history['logprobs'][sim_step] = logprobs 
                    history['values'][sim_step] = values     
                    history['generator'][sim_step] = torch.cuda.get_rng_state().clone()
                else:
                    actions, logprobs, values = [previous_history[key][sim_step] for key in ['actions', 'logprobs', 'values']]
                    torch.cuda.set_rng_state(previous_history['generator'][sim_step])
                                    

                ## Backward pass

                if not is_inference and sim_step>0:

                    returns = previous_history['returns'][sim_step]
                    advantages = previous_history['advantages'][sim_step]
                    new_logprobs, new_values, entropies, new_dist = model_student.evaluate(states, actions)

                    # Loss
                    ratios = torch.exp(new_logprobs - logprobs)
                    surr1 = ratios * advantages
                    surr2 = torch.clamp(ratios, 1-eps_clip, 1+eps_clip) * advantages
                    loss_actor = -(torch.min(surr1, surr2)).mean()
                    loss_entropy = -c_entropy * entropies.mean()
                    loss_critic = c_critic * F.mse_loss(new_values, returns)
                    loss =  (loss_actor + loss_critic + loss_entropy) /accumulation_steps
                    loss.backward()
                    backward_count += 1
                    kl_div += torch.distributions.kl_divergence(new_dist, old_dist).mean()/accumulation_steps


                    # Step optimizer

                    if backward_count % accumulation_steps == 0 and backward_count != 0:
                        torch.nn.utils.clip_grad_norm_(model_student.parameters(), c_grad_norm)
                        optimizer.step()
                        optimizer.zero_grad()

                        # Adaptive schedule
                        for param_group in optimizer.param_groups:
                            if adaptive_schedule:
                                param_group['lr'] = tr.update_lr(param_group['lr'], kl_div)

                        # Reinitialize
                        backward_count = 0
                        kl_div = 0


                    all_losses_actor.append(loss_actor.item())
                    all_losses_critic.append(loss_critic.item())

                # Update buffers

                actions_star = model.instanciate_actions(actions, dt=cf.frames_future)
                buffer['actions_star'] += list(torch.unbind( actions_star ))


            ## Current action (numEnvs, 1)

            if(len(buffer['actions_star'])>0):
                action = buffer['actions_star'].pop(0)
                action = torch.clamp(action, min=-1, max=1)
            else:
                action = torch.dzeros(numEnvs, 1)

            if sim_count % 6 == 0 and env.sim_frame_count % sim_duration == 256:
                torchvision.io.write_video('/isaac-sim/memoryPerceiver/memory_perceiver/main/monitor/test_'  + str(sim_count) + '_vid_.mp4', 
                                            torch.stack(full_video)[:,1], 4*cf.frames)

            # Step environment

            if sim_step == 0:
                env._task.post_reset()
            env._task.pre_physics_step(action.to(task.rl_device))
            env._world.step(render=render)
            env.sim_frame_count += 1
            env._task.post_physics_step()

            sim_step+=1

        else:
            env._world.step(render=render)

    env._simulation_app.close()


if __name__ == '__main__':
    sim_duration = 260
    numEnvs = cf.batch_size
    learning_rate = 1e-4
    accumulation_steps = 8
    c_entropy = 0.01
    c_critic=0.5
    student_update_step = 1
    eps=1e-9
    c_gamma=0.99
    c_lambda=0.95
    c_grad_norm=1.0
    eps_clip=0.2
    adaptive_schedule=False
    path_model = 'weights/humanoid.pt'
    with initialize(config_path="memoryPerceiver/OmniIsaacGymEnvs/omniisaacgymenvs/cfg"):
        cfg = compose(config_name="config", overrides=["task=Humanoid", "headless=True","enable_viewport=True",
                        "+numEnvs="+str(numEnvs)])
    parse_hydra_configs(cfg)

/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config': Defaults list is missing `_self_`. See https://hydra.cc/docs/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:412: UserWarning: In config: Invalid overriding of hydra/job_logging:
Default list overrides requires 'override' keyword.
See https://hydra.cc/docs/next/upgrades/1.0_to_1.1/defaults_list_override for more information.

  deprecation_warning(msg)


Starting kit application with the following args:  ['/isaac-sim/exts/omni.isaac.kit/omni/isaac/kit/simulation_app.py', '/isaac-sim/apps/omni.isaac.sim.python.gym.headless.render.kit', '--/app/tokens/exe-path=/isaac-sim/kit', '--/persistent/app/viewport/displayOptions=3094', '--/rtx/materialDb/syncLoads=True', '--/rtx/hydra/materialSyncLoads=True--/omni.kit.plugin/syncUsdLoads=True', '--/app/renderer/resolution/width=1280', '--/app/renderer/resolution/height=720', '--/app/window/width=1440', '--/app/window/height=900', '--/renderer/multiGpu/enabled=True', '--/app/fastShutdown=False', '--ext-folder', '/isaac-sim/exts', '--ext-folder', '/isaac-sim/apps', '--/physics/cudaDevice=0', '--portable', '--no-window', '--allow-root']
Passing the following args to the base kit application:  ['-f', '/root/.local/share/jupyter/runtime/kernel-9d2b686a-859a-427a-82f9-dafae9653b99.json']
[Warning] [omni.isaac.kit.simulation_app] Modules: ['omni.isaac.kit.app_framework'] were loaded before SimulationApp 

/isaac-sim/kit/kernel/py/omni/kit/app/_impl/__init__.py:104: DeprecationWarning: Using or importing the ABCs from 'collections' instead of from 'collections.abc' is deprecated since Python 3.3,and in 3.9 it will stop working
  if not isinstance(setting_py, str) and isinstance(setting_py, collections.Sequence):


xt v1.0],[carb::windowing::IWindowing v1.3]) (impl: carb.windowing-glfw.plugin)
2024-03-14 09:02:40 [1,314ms] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2024-03-14 09:02:40 [1,314ms] [Warning] [carb] Failed to startup plugin carb.windowing-glfw.plugin (interfaces: [carb::windowing::IGLContext v1.0],[carb::windowing::IWindowing v1.3]) (impl: carb.windowing-glfw.plugin)
[1.330s] [ext: omni.kit.renderer.capture-0.0.0] startup
[1.332s] [ext: omni.kit.renderer.imgui-0.0.0] startup
2024-03-14 09:02:40 [1,324ms] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2024-03-14 09:02:40 [1,324ms] [Warning] [carb] Failed to startup plugin carb.windowing-glfw.plugin (interfaces: [carb::windowing::IGLContext v1.0],[carb::windowing::IWindowing v1.3]) (impl: carb.windowing-glfw.plugin)
2024-03-14 09:02:40 [1,325ms] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2024-03-14 09:02:40 [1,325ms] [Warning] [carb] Failed to startup plugin carb.window

/isaac-sim/kit/exts/omni.graph/omni/graph/core/_impl/autonode/type_definitions.py:12: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  Float = NewType("Float", np.float)


Warp 0.6.3 initialized:
   CUDA Toolkit: 11.5, Driver: 12.2
   Devices:
     "cpu"    | x86_64
     "cuda:0" | NVIDIA GeForce RTX 3060 (sm_86)
   Kernel cache: /root/.cache/warp/0.6.3
0.10] startup
[2.127s] [ext: omni.kit.hydra_texture-1.0.11] startup
[2.136s] [ext: omni.kit.context_menu-1.5.12] startup
[2.140s] [ext: omni.ui.scene-1.5.18] startup
[2.146s] [ext: omni.kit.window.drop_support-1.0.1] startup
[2.147s] [ext: omni.kit.widget.viewport-104.1.16] startup
[2.150s] [ext: omni.kit.window.preferences-1.3.8] startup
[2.219s] [ext: omni.usdphysics-104.2.4-5.1] startup
[2.224s] [ext: omni.kit.viewport.window-104.1.23] startup
2024-03-14 09:02:41 [2,218ms] [Warning] [omni.kit.ui] Failed to set value for menu with path: Window/Viewport/Viewport 1. It doesn't exist.
[2.258s] [ext: omni.kit.manipulator.transform-104.6.12] startup
[2.260s] [ext: omni.physx-104.2.4-5.1] startup
[2.295s] [ext: omni.kit.viewport.utility-1.0.14] startup
[2.296s] [ext: omni.kit.widget.toolbar-1.4.0] startup
[2.

/isaac-sim/kit/python/lib/python3.7/site-packages/gym/spaces/box.py:84: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


[6.806s] app ready
[6.900s] Simulation App Startup Complete
[6.915s] [ext: omni.usd.schema.isaac-1.1.0] startup
[6.951s] [ext: omni.inspect-1.0.1] startup
[6.955s] [ext: omni.graph.action-1.31.1] startup
[6.969s] [ext: omni.graph.visualization.nodes-2.0.0] startup
[6.982s] [ext: omni.kit.widget.graph-1.5.6-104_2] startup
[6.996s] [ext: omni.kit.window.property-1.8.2] startup
[6.999s] [ext: omni.kit.window.extensions-1.1.1] startup
[7.006s] [ext: omni.syntheticdata-0.2.4] startup
2024-03-14 09:02:46 [7,012ms] [Warning] [omni.kit.viewport.menubar.core.menu_item.viewport_menu_item] ViewportMenuItem order argument is deprecated, use order_setting_path
2024-03-14 09:02:46 [7,012ms] [Warning] [omni.syntheticdata.scripts.extension] SyntheticData extension needs at least a stageFrameHistoryCount of 3
[7.030s] [ext: omni.isaac.ui-0.8.0] startup
[7.034s] [ext: omni.debugdraw-0.1.1] startup
[7.042s] [ext: usdrt.scenegraph-4.3.2] startup
[7.067s] [ext: omni.isaac.debug_draw-0.2.3] startup
[7.076s]

tensor([ 0.1695, -0.0433,  0.0740, -0.0924,  0.1907,  0.2727,  0.2531,  0.3852,
         0.1432,  0.1579,  0.5890, -0.0768,  0.3247,  0.1383,  0.1056,  0.2572,
         0.0312,  0.3800, -0.0234, -0.1342,  0.4375], device='cuda:0')
tensor([-0.4173,  0.9456, -0.0703, -0.1611,  0.3591,  0.0134,  0.5585, -0.1176,
        -0.1720, -0.1692,  0.3574, -0.3365, -0.0205,  0.4142, -0.2146,  0.2547,
        -0.2509, -0.0432,  0.5172, -0.1821,  0.0469], device='cuda:0')
tensor([ 0.3262, -0.1004,  0.3687, -0.2099, -0.1008,  0.3116,  0.4468,  0.3015,
         0.2289, -0.3202,  0.6538,  0.1484, -0.1722,  0.3080,  0.0828,  0.2390,
        -0.3015, -0.1010,  0.1141, -0.0562,  0.1625], device='cuda:0')
2024-03-14 09:21:21 [1,122,723ms] [Warning] [carb] Plugin interface for a client: omni.hydratexture.plugin was already released.
2024-03-14 09:21:21 [1,122,723ms] [Warning] [carb] Plugin interface for a client: omni.hydratexture.plugin was already released.
2024-03-14 09:21:21 [1,122,723ms] [Warning] [carb

In [ ]:
!nvidia-smi

: 